# Score-Based Perspective: From EBMs to NCSN

## 0. Prelim

As before, we have a set of $N$ data samples $\mathcal X=\{x_i|i\in \mathbb Z_N \}$, assumed uniquely sampled from an unknown distribution $x_i\sim p_{\text{data}}$.

The MNIST dataset is a good example. In this case, $x_i\in\mathbb R^{784}$.

In the previous chapter, we approached generative modelling from the **variational perspective** — learning a latent variable model by maximizing the ELBO. Now we take a fundamentally different approach: instead of modelling the density $p(x)$ directly, we model its **score function** $\nabla_x \log p(x)$.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])

import os
os.makedirs('../0.Data/', exist_ok=True)
train_data = datasets.MNIST('../0.Data/', train=True, download=True, transform=transform)
test_data = datasets.MNIST('../0.Data/', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

print(f"Loaded {len(train_data)} train, {len(test_data)} test samples")

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    img = images[i].view(28, 28)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'{labels[i]}')
    axes[i].axis('off')
plt.show()

print(f"Batch shape: {images.shape}")
print(f"Single image shape: {images[0].shape}")

## 1. Modelling

### 1.1 Energy-Based Models (EBMs)

An Energy-Based Model defines a probability distribution through an **energy function** $E_\theta(x):\mathbb R^D \to \mathbb R$:
$$p_\theta(x) = \frac{\exp(-E_\theta(x))}{Z_\theta}, \quad \text{where } Z_\theta = \int \exp(-E_\theta(x))\,dx$$

The key challenge: the partition function $Z_\theta$ is **intractable** for high-dimensional data. This means we cannot:
1. Evaluate $p_\theta(x)$ exactly (needed for MLE)
2. Sample from $p_\theta(x)$ directly

For maximum likelihood training, we would need:
$$\nabla_\theta \log p_\theta(x) = -\nabla_\theta E_\theta(x) - \nabla_\theta \log Z_\theta$$

The second term requires computing $\mathbb E_{p_\theta}[\nabla_\theta E_\theta(x)]$, which involves sampling from $p_\theta$ — creating a chicken-and-egg problem.

Traditional approaches (Contrastive Divergence, Persistent CD) use MCMC to approximately sample from $p_\theta$ during training, but this is slow and often unreliable in high dimensions.

### 1.2 From Energy-Based to Score-Based Generative Models

The key insight: if we take the gradient of $\log p_\theta(x)$ with respect to $x$ (not $\theta$), the partition function **vanishes**:
$$\nabla_x \log p_\theta(x) = \nabla_x \left[-E_\theta(x) - \log Z_\theta\right] = -\nabla_x E_\theta(x)$$

This gradient $\nabla_x \log p(x)$ is called the **score function** (or Stein score). It is a vector field $s(x):\mathbb R^D \to \mathbb R^D$ that points in the direction of increasing log-density at every point in space.

**Core idea:** Instead of learning $p_\theta(x)$ (which requires $Z_\theta$), learn a **score network** $s_\theta(x) \approx \nabla_x \log p_{\text{data}}(x)$.

Once we have the score function, we can generate samples via **Langevin dynamics**:
$$x_{k+1} = x_k + \frac{\eta}{2} \nabla_x \log p(x_k) + \sqrt{\eta}\, z_k, \quad z_k \sim \mathcal N(0, \mathbf I)$$

As $\eta \to 0$ and $K \to \infty$, the iterates $x_K$ converge to a sample from $p(x)$. The score function serves as the "drift" that guides random walkers toward high-density regions.

### 1.3 Score Matching

How do we train $s_\theta(x)$ to approximate $\nabla_x \log p_{\text{data}}(x)$? The natural objective is:
$$\mathcal L_{\text{ESM}}(\theta) = \frac{1}{2}\mathbb E_{p_{\text{data}}}\left[\|s_\theta(x) - \nabla_x \log p_{\text{data}}(x)\|^2_2\right]$$

But we don't know $\nabla_x \log p_{\text{data}}(x)$! This is where **score matching** (Hyvärinen, 2005) comes in.

#### Explicit (Vanilla) Score Matching

Through integration by parts, the objective can be rewritten without $\nabla_x \log p_{\text{data}}$:
$$\mathcal L_{\text{SM}}(\theta) = \mathbb E_{p_{\text{data}}}\left[\text{tr}(\nabla_x s_\theta(x)) + \frac{1}{2}\|s_\theta(x)\|^2_2\right] + C$$

The first term $\text{tr}(\nabla_x s_\theta(x))$ is the trace of the Jacobian of the score network — computationally expensive ($O(D)$ backward passes) for high-dimensional data.

#### Sliced Score Matching

Projects the score to random directions $v$ to avoid the full Jacobian:
$$\mathcal L_{\text{SSM}}(\theta) = \mathbb E_{p_v}\mathbb E_{p_{\text{data}}}\left[v^T \nabla_x s_\theta(x)\, v + \frac{1}{2}(v^T s_\theta(x))^2\right]$$

This only requires a single Jacobian-vector product (efficient via autodiff), but is still noisy.

### 1.4 Denoising Score Matching (DSM)

The most practical variant. Instead of matching the score of $p_{\text{data}}$, we match the score of a **noise-perturbed** distribution.

Given a noise kernel $q_\sigma(\tilde x | x) = \mathcal N(\tilde x; x, \sigma^2 \mathbf I)$, the perturbed distribution is:
$$q_\sigma(\tilde x) = \int q_\sigma(\tilde x | x)\, p_{\text{data}}(x)\, dx$$

**Key result** (Vincent, 2011): minimizing the denoising score matching objective is equivalent to the explicit score matching objective on the perturbed distribution:
$$\mathcal L_{\text{DSM}}(\theta; \sigma) = \frac{1}{2}\mathbb E_{p_{\text{data}}(x)}\mathbb E_{q_\sigma(\tilde x|x)}\left[\|s_\theta(\tilde x) - \nabla_{\tilde x} \log q_\sigma(\tilde x | x)\|^2_2\right]$$

Since $q_\sigma(\tilde x | x) = \mathcal N(\tilde x; x, \sigma^2 \mathbf I)$, the conditional score has a **closed form**:
$$\nabla_{\tilde x} \log q_\sigma(\tilde x | x) = -\frac{\tilde x - x}{\sigma^2} = -\frac{\epsilon}{\sigma}, \quad \text{where } \epsilon = \frac{\tilde x - x}{\sigma} \sim \mathcal N(0, \mathbf I)$$

So the DSM loss becomes:
$$\mathcal L_{\text{DSM}}(\theta; \sigma) = \frac{1}{2}\mathbb E_{x \sim p_{\text{data}}}\mathbb E_{\epsilon \sim \mathcal N(0, \mathbf I)}\left[\left\|s_\theta(x + \sigma\epsilon) + \frac{\epsilon}{\sigma}\right\|^2_2\right]$$

**Interpretation:** We're training the network to predict the direction back to the clean data from noisy observations — the "denoising" direction. No Jacobian computation needed!

### 1.5 Multi-Noise Denoising Score Matching: NCSN

A single noise level $\sigma$ creates a dilemma:
- **Small $\sigma$**: accurate score estimation near the data manifold, but Langevin dynamics mixes slowly (gets trapped in modes)
- **Large $\sigma$**: good global structure and fast mixing, but inaccurate score near the data

**NCSN** (Song & Ermon, 2019) resolves this with a **geometric sequence** of noise levels:
$$\sigma_1 > \sigma_2 > \cdots > \sigma_L, \quad \text{where } \sigma_i = \sigma_1 \left(\frac{\sigma_L}{\sigma_1}\right)^{\frac{i-1}{L-1}}$$

A single **Noise Conditional Score Network** $s_\theta(x, \sigma)$ is trained across all noise levels:
$$\mathcal L_{\text{NCSN}}(\theta) = \frac{1}{L}\sum_{i=1}^{L} \lambda(\sigma_i)\, \mathbb E_{p_{\text{data}}(x)}\mathbb E_{\tilde x \sim \mathcal N(x, \sigma_i^2 \mathbf I)}\left[\left\|s_\theta(\tilde x, \sigma_i) + \frac{\tilde x - x}{\sigma_i^2}\right\|^2_2\right]$$

The weighting $\lambda(\sigma_i) = \sigma_i^2$ is chosen so that each noise level contributes roughly equally to the loss (since the score magnitude scales as $1/\sigma$).

#### Annealed Langevin Dynamics

At inference, we run Langevin dynamics starting from the **largest** noise level and progressively **annealing** to smaller ones:

For $i = 1, 2, \ldots, L$:
$$x_{k+1} = x_k + \frac{\eta_i}{2}\, s_\theta(x_k, \sigma_i) + \sqrt{\eta_i}\, z_k, \quad z_k \sim \mathcal N(0, \mathbf I)$$

where $\eta_i = \alpha \cdot \sigma_i^2 / \sigma_L^2$ for some small $\alpha > 0$.

**Intuition:** At high noise, the score field provides broad, global guidance ("move toward the data manifold"). As noise decreases, the score provides finer, local detail ("refine into a specific sample"). This is conceptually analogous to the reverse diffusion process in DDPMs.

### 1.6 Summary: A Comparative View of NCSN and DDPM

| Aspect | DDPM | NCSN |
|--------|------|------|
| **Perspective** | Variational (ELBO) | Score-based |
| **What is learned** | Noise predictor $\epsilon_\phi(x_i, i)$ | Score network $s_\theta(x, \sigma)$ |
| **Perturbation** | Markov chain $q(x_i|x_{i-1})$ | Direct noise $q_\sigma(\tilde x|x) = \mathcal N(\tilde x; x, \sigma^2 I)$ |
| **Loss** | $\|\epsilon_\phi(x_i, i) - \epsilon\|^2$ | $\|s_\theta(\tilde x, \sigma) + \epsilon/\sigma\|^2$ |
| **Sampling** | Ancestral (reverse Markov chain) | Annealed Langevin dynamics |
| **Schedule** | $\{\beta_i\}$ (linear/cosine) | $\{\sigma_i\}$ (geometric) |

**Deep connection:** The DDPM noise prediction $\epsilon_\phi$ and the NCSN score $s_\theta$ are related by:
$$s_\theta(x_i, i) = -\frac{\epsilon_\phi(x_i, i)}{\sqrt{1 - \bar\alpha_i^2}}$$

Both frameworks were later unified under the **Stochastic Differential Equation** (SDE) framework (Song et al., 2021), which views diffusion as a continuous-time process — the subject of the next chapter.

---

Now let's implement NCSN from scratch on MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

### Noise Schedule

We define a geometric sequence of $L$ noise levels from $\sigma_1$ (large) to $\sigma_L$ (small):
$$\sigma_i = \sigma_1 \left(\frac{\sigma_L}{\sigma_1}\right)^{\frac{i-1}{L-1}}$$

In [ ]:
class NoiseScheduleNCSN:
    def __init__(self, sigma_begin=1.0, sigma_end=0.01, n_levels=10, device='cuda'):
        self.n_levels = n_levels
        self.device = device
        
        # Geometric sequence of noise levels
        self.sigmas = torch.exp(
            torch.linspace(np.log(sigma_begin), np.log(sigma_end), n_levels)
        ).to(device)
    
    def __repr__(self):
        return f"NoiseScheduleNCSN(sigmas={self.sigmas.cpu().tolist()})"


schedule = NoiseScheduleNCSN(sigma_begin=1.0, sigma_end=0.01, n_levels=10, device='cpu')
print(schedule)
print(f"\nNoise levels (sigma_1 to sigma_L):")
for i, s in enumerate(schedule.sigmas):
    print(f"  sigma_{i+1:2d} = {s:.4f}")

### Score Network Architecture

The score network $s_\theta(x, \sigma): \mathbb R^D \times \mathbb R_+ \to \mathbb R^D$ takes a (possibly noisy) data point and a noise level, and outputs a vector of the same dimension as $x$.

We condition on $\sigma$ by passing it through a small embedding MLP and adding it to the hidden representations — analogous to time conditioning in DDPMs.

We use the same UNet architecture from the DDPM notebook, modified to condition on $\sigma$ instead of timestep $t$.

In [ ]:
class ScoreNet(nn.Module):
    """UNet-based score network s_theta(x, sigma) for NCSN.
    
    Conditions on the noise level sigma via a learned embedding,
    analogous to time conditioning in DDPM.
    """
    def __init__(self, in_channels=1, out_channels=1, sigma_embedding_dim=64, hidden_dims=[32, 32, 64]):
        super().__init__()
        self.sigma_embedding_dim = sigma_embedding_dim
        
        # Sigma embedding MLP (log(sigma) -> embedding)
        self.sigma_mlp = nn.Sequential(
            nn.Linear(sigma_embedding_dim, sigma_embedding_dim * 4),
            nn.SiLU(),
            nn.Linear(sigma_embedding_dim * 4, sigma_embedding_dim * 4)
        )
        
        # Encoder
        self.enc1 = self._make_block(in_channels, hidden_dims[0])
        self.enc2 = self._make_block(hidden_dims[0], hidden_dims[1])
        self.enc3 = self._make_block(hidden_dims[1], hidden_dims[2])
        
        # Bottleneck
        self.bottleneck = self._make_block(hidden_dims[2], hidden_dims[2])
        
        # Decoder with skip connections
        self.dec3 = self._make_block(hidden_dims[2] * 2, hidden_dims[1])
        self.dec2 = self._make_block(hidden_dims[1] * 2, hidden_dims[0])
        self.dec1 = self._make_block(hidden_dims[0] * 2, hidden_dims[0])
        
        # Final output
        self.final = nn.Conv2d(hidden_dims[0], out_channels, kernel_size=1)
        
        # Sigma projection layers for each level
        self.sigma_projs = nn.ModuleList([
            nn.Linear(sigma_embedding_dim * 4, hidden_dims[0]),
            nn.Linear(sigma_embedding_dim * 4, hidden_dims[1]),
            nn.Linear(sigma_embedding_dim * 4, hidden_dims[2])
        ])
    
    def _make_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU()
        )
    
    def get_sigma_embedding(self, sigma, channels):
        """Sinusoidal positional embedding for log(sigma)."""
        log_sigma = torch.log(sigma)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2, device=sigma.device) / channels))
        pos_enc_a = torch.sin(log_sigma.unsqueeze(1) * inv_freq)
        pos_enc_b = torch.cos(log_sigma.unsqueeze(1) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=1)
        return pos_enc
    
    def forward(self, x, sigma):
        """Forward pass.
        
        Args:
            x: [B, 1, 28, 28] noisy image
            sigma: [B] noise level for each sample
        Returns:
            score: [B, 1, 28, 28] estimated score
        """
        # Get sigma embeddings
        s_emb = self.get_sigma_embedding(sigma, self.sigma_embedding_dim)
        s_emb = self.sigma_mlp(s_emb)
        
        # Encoder with sigma conditioning
        e1 = self.enc1(x)
        e1 = e1 + self.sigma_projs[0](s_emb)[:, :, None, None]
        
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e2 = e2 + self.sigma_projs[1](s_emb)[:, :, None, None]
        
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e3 = e3 + self.sigma_projs[2](s_emb)[:, :, None, None]
        
        # Bottleneck
        b = self.bottleneck(F.max_pool2d(e3, 2))
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([F.interpolate(b, e3.shape[2:], mode='nearest'), e3], dim=1))
        d2 = self.dec2(torch.cat([F.interpolate(d3, e2.shape[2:], mode='nearest'), e2], dim=1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, e1.shape[2:], mode='nearest'), e1], dim=1))
        
        # Output: predicted score
        score = self.final(d1)
        return score


# Quick test
test_model = ScoreNet().to('cpu')
test_x = torch.randn(2, 1, 28, 28)
test_sigma = torch.tensor([0.5, 0.1])
test_out = test_model(test_x, test_sigma)
print(f"Input:  {test_x.shape}")
print(f"Output: {test_out.shape}")
print(f"Parameters: {sum(p.numel() for p in test_model.parameters()):,}")

### NCSN Loss

The weighted denoising score matching loss across all noise levels:
$$\mathcal L_{\text{NCSN}}(\theta) = \frac{1}{L}\sum_{i=1}^{L} \sigma_i^2 \cdot \mathbb E_{x}\mathbb E_{\epsilon}\left[\left\|s_\theta(x + \sigma_i\epsilon,\, \sigma_i) + \frac{\epsilon}{\sigma_i}\right\|^2_2\right]$$

In practice, we sample a random noise level per batch element.

In [ ]:
class NCSN:
    """Noise Conditional Score Network."""
    
    def __init__(self, model, noise_schedule, device='cuda'):
        self.model = model
        self.noise_schedule = noise_schedule
        self.device = device
    
    def dsm_loss(self, x_0):
        """Compute the denoising score matching loss.
        
        For each sample in the batch, randomly pick a noise level sigma_i,
        perturb x_0 with that noise, and train the score network to predict
        the score of the perturbed distribution.
        
        Args:
            x_0: [B, 1, 28, 28] clean images
        Returns:
            loss: scalar
        """
        batch_size = x_0.shape[0]
        sigmas = self.noise_schedule.sigmas
        
        # Sample random noise level index for each sample
        idx = torch.randint(0, len(sigmas), (batch_size,), device=self.device)
        sigma = sigmas[idx]  # [B]
        
        # Sample noise and perturb
        epsilon = torch.randn_like(x_0)  # [B, 1, 28, 28]
        x_tilde = x_0 + sigma[:, None, None, None] * epsilon  # [B, 1, 28, 28]
        
        # Predict score
        score = self.model(x_tilde, sigma)  # [B, 1, 28, 28]
        
        # Target score: -epsilon / sigma
        target = -epsilon / sigma[:, None, None, None]
        
        # Weighted loss: lambda(sigma) = sigma^2
        # ||s_theta(x_tilde, sigma) - (-epsilon/sigma)||^2 * sigma^2
        loss_per_sample = (score - target).pow(2).sum(dim=(1, 2, 3))  # [B]
        weight = sigma.pow(2)  # [B]
        
        loss = (weight * loss_per_sample).mean()
        return loss
    
    @torch.no_grad()
    def annealed_langevin_dynamics(self, batch_size, img_shape, n_steps_per_level=100, step_lr=2e-5):
        """Generate samples via Annealed Langevin Dynamics.
        
        Start from noise, iterate through noise levels from largest to smallest,
        running Langevin dynamics at each level.
        
        Args:
            batch_size: number of samples to generate
            img_shape: (C, H, W)
            n_steps_per_level: Langevin steps per noise level
            step_lr: base step size alpha
        Returns:
            x: [B, C, H, W] generated samples
            intermediates: list of intermediate samples for visualization
        """
        self.model.eval()
        sigmas = self.noise_schedule.sigmas
        
        # Initialize from uniform noise (or large Gaussian)
        x = torch.rand(batch_size, *img_shape, device=self.device)
        
        intermediates = [x.cpu().clone()]
        
        for i, sigma in enumerate(tqdm(sigmas, desc='Annealing')):
            # Step size: eta_i = alpha * (sigma_i / sigma_L)^2
            eta = step_lr * (sigma / sigmas[-1]).pow(2)
            
            sigma_batch = sigma.expand(batch_size)  # [B]
            
            for k in range(n_steps_per_level):
                # Score estimate
                score = self.model(x, sigma_batch)
                
                # Langevin update
                noise = torch.randn_like(x)
                x = x + (eta / 2) * score + torch.sqrt(eta) * noise
            
            intermediates.append(x.cpu().clone())
        
        return x, intermediates

## 2. Train

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Hyperparameters
sigma_begin = 1.0
sigma_end = 0.01
n_levels = 10
lr = 1e-4
train_epochs = 100

# Initialize
noise_schedule = NoiseScheduleNCSN(sigma_begin, sigma_end, n_levels, device=device)
model = ScoreNet(in_channels=1, out_channels=1).to(device)
ncsn = NCSN(model, noise_schedule, device=device)
optimizer = optim.Adam(model.parameters(), lr=lr)

print(f"Noise levels: {noise_schedule.sigmas.cpu().tolist()}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
losses = []

for epoch in range(train_epochs):
    model.train()
    train_loss = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        # Reshape to [B, 1, 28, 28] and normalize to [0, 1]
        data = data.view(-1, 1, 28, 28).to(device)
        
        optimizer.zero_grad()
        loss = ncsn.dsm_loss(data)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    avg_loss = train_loss / len(train_loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{train_epochs}, Average Loss: {avg_loss:.4f}')

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('DSM Loss')
plt.title('NCSN Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Inference

We generate samples via **Annealed Langevin Dynamics**: starting from random noise, we iteratively refine using the learned score, annealing through noise levels from $\sigma_1$ (largest) to $\sigma_L$ (smallest).

In [ ]:
# Generate samples
n_samples = 8
print(f"Generating {n_samples} samples via Annealed Langevin Dynamics...")

samples, intermediates = ncsn.annealed_langevin_dynamics(
    batch_size=n_samples,
    img_shape=(1, 28, 28),
    n_steps_per_level=100,
    step_lr=2e-5
)

# Visualize final samples
samples_viz = samples.clamp(0, 1)

fig, axes = plt.subplots(1, n_samples, figsize=(20, 3))
for i in range(n_samples):
    axes[i].imshow(samples_viz[i, 0].cpu(), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'Sample {i+1}')
plt.suptitle('Generated Samples from NCSN (Annealed Langevin Dynamics)')
plt.show()

In [ ]:
# Visualize the annealing process across noise levels
print("Visualizing the annealing process...")

n_show = min(len(intermediates), 11)  # init + one per sigma level
step_indices = np.linspace(0, len(intermediates) - 1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
sigmas_list = [float('inf')] + noise_schedule.sigmas.cpu().tolist()

for idx, step_i in enumerate(step_indices):
    img = intermediates[step_i][0, 0].clamp(0, 1)
    axes[idx].imshow(img, cmap='gray')
    axes[idx].axis('off')
    if step_i == 0:
        axes[idx].set_title('Init (noise)', fontsize=9)
    else:
        axes[idx].set_title(f'$\\sigma={sigmas_list[step_i]:.3f}$', fontsize=9)

plt.suptitle('Annealed Langevin Dynamics: From Noise to Image', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the score field at different noise levels for a single sample
print("Visualizing score fields at different noise levels...")

model.eval()
test_img = next(iter(test_loader))[0][0].view(1, 1, 28, 28).to(device)

fig, axes = plt.subplots(2, noise_schedule.n_levels + 1, figsize=(3 * (noise_schedule.n_levels + 1), 6))

# First column: clean image
axes[0, 0].imshow(test_img[0, 0].cpu(), cmap='gray')
axes[0, 0].set_title('Clean', fontsize=9)
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for i, sigma in enumerate(noise_schedule.sigmas):
    # Perturb
    noise = torch.randn_like(test_img)
    x_noisy = test_img + sigma * noise
    
    # Get score
    with torch.no_grad():
        score = model(x_noisy, sigma.unsqueeze(0))
    
    # Plot noisy image
    axes[0, i+1].imshow(x_noisy[0, 0].cpu().clamp(0, 1), cmap='gray')
    axes[0, i+1].set_title(f'$\\sigma={sigma:.3f}$', fontsize=9)
    axes[0, i+1].axis('off')
    
    # Plot score magnitude
    score_mag = score[0, 0].cpu().abs()
    axes[1, i+1].imshow(score_mag, cmap='hot')
    axes[1, i+1].set_title(f'|score|', fontsize=9)
    axes[1, i+1].axis('off')

axes[0, 0].set_ylabel('Noisy Image', fontsize=10)
axes[1, 0].set_ylabel('Score Magnitude', fontsize=10)
plt.suptitle('Score Fields at Different Noise Levels', fontsize=13)
plt.tight_layout()
plt.show()